# Processing the Satellite data downloaded from Google Drive to add lat, long as columns

In [1]:
import os
import pandas as pd

import yaml

# Load Configuration
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

# DATA_DIR = "./shc_data/"
# SATELLITE_DIR = "SATELLITE_DATA"
# YEAR = "AGRI_2023-24"

DATA_DIR = config['paths']['data_dir']
SATELLITE_DIR = config['paths']['satellite_dir']
YEAR = config['paths']['output_sub_dir']

PATH = os.path.join(DATA_DIR, SATELLITE_DIR, YEAR)
# states = os.listdir(PATH)
states = [
    d for d in os.listdir(PATH)
    if os.path.isdir(os.path.join(PATH, d))
]

state_dfs = []
for state in states:
    district_dfs = []
    # files = os.listdir(os.path.join(PATH, state))
    state_path = os.path.join(PATH, state)
    files = [
        f for f in os.listdir(state_path)
        if f.endswith('.csv')
    ]
    for file in files:
        try:
            df = pd.read_csv(os.path.join(PATH, state, file))
        except pd.errors.EmptyDataError:
            continue
        district_dfs.append(df) 
    try:
        temp_df = pd.concat(district_dfs, ignore_index=True)
    except ValueError:
        continue
    
    state_dfs.append(temp_df)

merged_df = pd.concat(state_dfs, ignore_index=True)

# Apply your split logic and extract lat/lon
coords = merged_df['.geo'].apply(lambda x: x.split('"coordinates":')[1].split('}')[0].strip('[').strip(']').split(','))

# Convert to DataFrame and assign to new columns
merged_df[['longitude', 'latitude']] = pd.DataFrame(coords.tolist(), index=merged_df.index).astype(float)
merged_df = merged_df.drop(['.geo'], axis=1)

print(merged_df.head())

/var/folders/22/l33mmp4d2cd_rj_lchy4dhl40000gn/T/ipykernel_86924/1021666516.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  temp_df = pd.concat(district_dfs, ignore_index=True)
/var/folders/22/l33mmp4d2cd_rj_lchy4dhl40000gn/T/ipykernel_86924/1021666516.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  temp_df = pd.concat(district_dfs, ignore_index=True)
/var/folders/22/l33mmp4d2cd_rj_lchy4dhl40000gn/T/ipykernel_86924/1021666516.py:31: FutureWarning: The behavior of DataFrame concatenatio

  district     village        date           start_date             end_date  \
0   DEORIA  Singhi Dih  2024-02-25  2023-07-01T00:00:00  2024-06-30T00:00:00   
1   DEORIA   Maheshpur  2024-02-24  2023-07-01T00:00:00  2024-06-30T00:00:00   
2   DEORIA    Salhapur  2024-03-16  2023-07-01T00:00:00  2024-06-30T00:00:00   
3   DEORIA      Pachma  2024-02-25  2023-07-01T00:00:00  2024-06-30T00:00:00   
4   DEORIA      Dumari  2024-03-16  2023-07-01T00:00:00  2024-06-30T00:00:00   

        N     P      K    B     Fe  ...  sand515  silt05  silt515  clay05  \
0  135.00  13.5  166.5  0.8   8.65  ...    320.0   399.0    408.0   271.0   
1   74.25  22.5  117.0  0.6   8.58  ...    301.0   429.0    435.0   262.0   
2  137.25  18.0  202.5  0.5  10.25  ...    321.0   389.0    398.0   281.0   
3  123.75   9.0  157.5  0.6   7.54  ...    302.0   429.0    432.0   265.0   
4  108.00  18.0  126.0  0.6   8.42  ...    319.0   421.0    423.0   249.0   

   clay515  NDVI_Kharif  NDVI_Rabi  NDVI_Zaid  longitude

In [2]:
# merged_df.to_csv(os.path.join(DATA_DIR, SATELLITE_DIR, YEAR, "COMBINED_2024.csv"), index=False)

# Split the Satellite data into AEZs

In [3]:
import numpy as np
from utils.segregate_by_aez import segregate_by_aez

SAVE_DIR = os.path.join(DATA_DIR, "AEZS", YEAR)
os.makedirs(SAVE_DIR, exist_ok=True)

In [4]:
aez_df = segregate_by_aez(merged_df)

In [5]:
unique_aezs = aez_df['ae_regcode'].unique()
unique_aezs = unique_aezs[~np.isnan(unique_aezs)]
unique_aezs

array([13.,  9.,  4., 11., 10., 14.,  2.,  5., 19.,  6., 16.,  8., 18.,
        3.,  7., 12., 15., 17., 20.])

In [8]:
for aez in unique_aezs:
    df = aez_df[aez_df['ae_regcode'] == aez]
    print(int(aez), len(df))
    df.to_csv(os.path.join(SAVE_DIR, f"AEZ_{int(aez)}_tagged.csv"), index=False)

13 205491
9 221566
4 406655
11 119880
10 119494
14 53482
2 150280
5 145078
19 27938
6 136431
16 21207
8 193102
18 73301
3 32789
7 24242
12 98234
15 357019
17 20745
20 199
